# <u>Proyecto</u>: Análisis estadístico de series temporales ambientales de reservas de museos para la conservación preventiva sostenible de obras de arte

## 01. Preprocesamiento y auditoría de calidad

Este notebook construye la versión horaria y trazable del conjunto de datos utilizada en los análisis posteriores. Su objetivo no es corregir ni completar artificialmente las series, sino documentar su estructura, detectar problemas de calidad y explicitar las decisiones de preparación.

Se parte de una copia de los datos originales, se normalizan nombres y fechas, se audita el eje temporal y la disponibilidad de cada sensor, y finalmente se exporta un archivo procesado junto con metadatos de control. 

<u>Notas importantes</u>: Los huecos se conservan como `NaN`, no se realiza imputación en esta etapa.

## 1. Configuración y rutas

Se cargan las librerías necesarias y se definen las rutas de entrada y salida. Mantener estas rutas concentradas al comienzo facilita la reproducibilidad y evita referencias dispersas a archivos dentro del notebook.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

In [14]:
RAW_PATH = Path("../data/raw/SIVORI_SMN_y_reserva.csv")

PROCESSED_DIR = Path("../data/processed")

METADATA_DIR = Path("../data/metadata")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

METADATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga y preservación de los datos originales

El archivo fuente (`SIVORI_SMN_y_reserva.csv`) reúne en una única tabla horaria los registros de los cuatro HOBO y de la estación del SMN. Al unificarlos se verificó que las marcas temporales coincidieran. La HR se unificó a un decimal según el criterio adoptado para el conjunto de datos, ya que algunos registros originales tenían dos. A partir de acá, ese archivo se trata como dato crudo.

Primero se carga el archivo fuente y se conserva una copia en memoria antes de realizar transformaciones. A continuación se inspeccionan dimensiones, tipos y primeras filas para verificar que el archivo tenga la estructura esperada.

In [15]:
df_raw = pd.read_csv(RAW_PATH, sep=",")

In [16]:
df_copy = df_raw.copy()

In [17]:
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 8905 entries, 0 to 8904
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Fecha c/1hr            8905 non-null   str    
 1   H1 T (°C)              8872 non-null   float64
 2   H1 HRCal ( %)          8872 non-null   float64
 3   H2 T (°C)              8903 non-null   float64
 4   H2 HRCal (%)           8903 non-null   float64
 5   H3 T (°C)              7823 non-null   float64
 6   H3 HRCal (%)           7823 non-null   float64
 7   H4 T (°C)              7603 non-null   float64
 8   H4 HRCal (%)           7603 non-null   float64
 9   SMN T (°C)             8905 non-null   float64
 10  SMN HR (%)             8905 non-null   float64
 11  SMN Prec. c/6 hs (mm)  8905 non-null   float64
dtypes: float64(11), str(1)
memory usage: 835.0 KB


In [18]:
df_copy.head()

,Fecha c/1hr,H1 T (°C),H1 HRCal ( %),H2 T (°C),H2 HRCal (%),H3 T (°C),H3 HRCal (%),H4 T (°C),H4 HRCal (%),SMN T (°C),SMN HR (%),SMN Prec. c/6 hs (mm)
0,29 de December de 2022 a las 14:00,25.61,60.0,26.12,58.0,26.39,57.0,26.00,60.0,26.6,49.0,0.0
1,29 de December de 2022 a las 15:00,26.02,61.0,26.12,57.0,26.80,57.0,25.90,59.0,26.7,48.0,0.0
2,29 de December de 2022 a las 16:00,26.02,60.0,26.12,57.0,26.80,56.0,25.92,60.0,27.3,51.0,0.0
3,29 de December de 2022 a las 17:00,26.02,60.0,26.54,57.0,26.80,56.0,25.97,60.0,26.8,51.0,0.0
4,29 de December de 2022 a las 18:00,26.02,60.0,26.12,58.0,26.80,57.0,25.92,60.0,26.4,53.0,0.0


## 3. Normalización del esquema y del tiempo

Se normalizan los nombres de variables y se verifica la presencia de las columnas esperadas. Luego se transforma la fecha a `datetime`, se ordenan los registros y se delimita el período efectivamente cubierto por el monitoreo. Estas operaciones definen la base temporal común para el resto del proyecto. Además, se redondea la humedad relativa a un decimal para unificar el esquema de los datos.

In [19]:
df_copy.columns = (df_copy.columns.str.strip().str.replace(r"\s+"," ",regex=True)
    .str.replace(r"\(\s*%\s*\)","(%)",regex=True)
    .str.replace(r"\(\s*°C\s*\)","(°C)",regex=True)
)

In [20]:
df_copy.columns

Index(['Fecha c/1hr', 'H1 T (°C)', 'H1 HRCal (%)', 'H2 T (°C)', 'H2 HRCal (%)',
       'H3 T (°C)', 'H3 HRCal (%)', 'H4 T (°C)', 'H4 HRCal (%)', 'SMN T (°C)',
       'SMN HR (%)', 'SMN Prec. c/6 hs (mm)'],
      dtype='str')

In [21]:
hr_cols = [
    'H1 HRCal (%)',
    'H2 HRCal (%)',
    'H3 HRCal (%)',
    'H4 HRCal (%)',
    'SMN HR (%)'
]

df_copy[hr_cols] = np.trunc(df_copy[hr_cols] * 10) / 10

In [22]:
FECHA_ORIGINAL = "Fecha c/1hr"

HOBOS = ["H1", "H2", "H3", "H4"]

TEMP_INTERIOR = [f"{hobo} T (°C)" for hobo in HOBOS]
HR_INTERIOR = [f"{hobo} HRCal (%)" for hobo in HOBOS]

TEMP_SMN = "SMN T (°C)"
HR_SMN = "SMN HR (%)"
PREC_SMN = "SMN Prec. c/6 hs (mm)"

VARIABLES_INTERIORES = TEMP_INTERIOR + HR_INTERIOR
VARIABLES_SMN = [TEMP_SMN, HR_SMN, PREC_SMN]
VARIABLES_NUMERICAS = VARIABLES_INTERIORES + VARIABLES_SMN

In [23]:
columnas_esperadas = [FECHA_ORIGINAL, *VARIABLES_NUMERICAS]

faltantes_columnas = sorted(set(columnas_esperadas).difference(df_copy.columns))

extras_columnas = sorted(set(df_copy.columns).difference(columnas_esperadas))

if faltantes_columnas:

    raise ValueError(
        f"Faltan columnas esperadas en el archivo: "
        f"{faltantes_columnas}"
    )


print(f"Las {len(columnas_esperadas)} columnas esperadas están presentes.")


if extras_columnas:

    print(f"Columnas adicionales, no utilizadas: {extras_columnas}")

Las 12 columnas esperadas están presentes.


In [24]:
fecha_limpia = (df_copy[FECHA_ORIGINAL].astype(str).str.strip().str.replace(" de ", " ", regex=False).str.replace(" a las ", " ", regex=False))

MESES = {
    "January": "01",
    "February": "02",
    "March": "03",
    "April": "04",
    "May": "05",
    "June": "06",
    "July": "07",
    "August": "08",
    "September": "09",
    "October": "10",
    "November": "11",
    "December": "12"
}

for mes, numero in MESES.items():

    fecha_limpia = (
        fecha_limpia
        .str.replace(
            mes,
            numero,
            regex=False
        )
    )


df_copy["fecha"] = pd.to_datetime(
    fecha_limpia,
    format="%d %m %Y %H:%M",
    errors="raise"
)

In [25]:
df_copy = (
    df_copy
    .sort_values("fecha")
    .reset_index(drop=True)
)


inicio_monitoreo = df_copy["fecha"].min()
fin_monitoreo = df_copy["fecha"].max()
dias_cubiertos = (fin_monitoreo - inicio_monitoreo).days


print(f"""Inicio del monitoreo : {inicio_monitoreo}
Fin del monitoreo    : {fin_monitoreo}
Período cubierto     : {dias_cubiertos} días
Registros            : {len(df_copy)}""")

Inicio del monitoreo : 2022-12-29 14:00:00
Fin del monitoreo    : 2024-01-04 14:00:00
Período cubierto     : 371 días
Registros            : 8905


## 4. Auditoría del eje temporal

Antes de reindexar se revisan duplicados y diferencias entre marcas temporales consecutivas. Después se construye una grilla horaria completa entre el primer y el último registro. La reindexación hace explícitos los instantes faltantes, pero **no los imputa**. La variable `registro_original` permite distinguir observaciones presentes en el archivo fuente de filas incorporadas únicamente para completar la grilla temporal.

In [26]:
duplicados_fecha = df_copy["fecha"].duplicated(keep=False)
n_duplicados = int(duplicados_fecha.sum())


print(f"Timestamps duplicados: {n_duplicados}")


if n_duplicados > 0:

    display(
        df_copy
        .loc[duplicados_fecha]
        .sort_values("fecha")
    )

Timestamps duplicados: 0


In [27]:
frecuencias_observadas = (
    df_copy["fecha"]
    .diff()
    .value_counts()
    .sort_index()
)


print("Intervalos entre timestamps consecutivos:")

for intervalo, n in frecuencias_observadas.items():
    print(f"  {intervalo.total_seconds() / 3600:g} h : {n}")

Intervalos entre timestamps consecutivos:
  1 h : 8904


In [28]:
grilla_esperada = pd.date_range(
    start=inicio_monitoreo,
    end=fin_monitoreo,
    freq="1h",
    name="fecha"
)

fechas_faltantes = grilla_esperada.difference(df_copy["fecha"])


print(f"""Timestamps esperados  : {len(grilla_esperada)}
Timestamps observados : {len(df_copy)}
Horas ausentes        : {len(fechas_faltantes)}""")


if len(fechas_faltantes) > 0:
    display(pd.DataFrame({"fecha_faltante": fechas_faltantes}))

Timestamps esperados  : 8905
Timestamps observados : 8905
Horas ausentes        : 0


In [29]:
df_procesado = (
    df_copy
    .drop(columns=FECHA_ORIGINAL)
    .set_index("fecha")
    .reindex(grilla_esperada)
)

n_filas_agregadas = len(fechas_faltantes)


if n_filas_agregadas > 0:

    df_procesado["registro_original"] = ~df_procesado.index.isin(fechas_faltantes)

    print(
        f"Se agregaron {n_filas_agregadas} filas para completar la grilla "
        "horaria, marcadas con registro_original = False."
    )

else:

    print("La grilla horaria del archivo ya estaba completa: no se agregó ninguna fila.")

La grilla horaria del archivo ya estaba completa: no se agregó ninguna fila.


## 5. Tipos, faltantes y disponibilidad por sensor

Se verifica que las variables de medición sean numéricas y se resume la cantidad de valores faltantes. También se construyen indicadores de disponibilidad por HOBO y se comprueba si temperatura y humedad relativa faltan de manera consistente dentro de cada equipo.

La precipitación requiere una lectura diferente: los datos disponibles corresponden a acumulaciones en intervalos más largos, por lo que la ausencia de `NaN` no implica una medición horaria independiente.

In [30]:
print("Tipos de variables:")
display(df_procesado[VARIABLES_NUMERICAS].dtypes)


columnas_no_numericas = [
    col
    for col in VARIABLES_NUMERICAS
    if not pd.api.types.is_numeric_dtype(df_procesado[col])
]


if columnas_no_numericas:

    raise TypeError(
        f"Las siguientes variables deberían ser numéricas: {columnas_no_numericas}"
    )

Tipos de variables:


H1 T (°C)                float64
H2 T (°C)                float64
H3 T (°C)                float64
H4 T (°C)                float64
H1 HRCal (%)             float64
H2 HRCal (%)             float64
H3 HRCal (%)             float64
H4 HRCal (%)             float64
SMN T (°C)               float64
SMN HR (%)               float64
SMN Prec. c/6 hs (mm)    float64
dtype: object

In [31]:
numericas = df_procesado[VARIABLES_NUMERICAS]

missing_summary = pd.DataFrame({
    "n_observado": numericas.notna().sum(),
    "n_missing": numericas.isna().sum(),
    "pct_missing": numericas.isna().mean().mul(100),
    "pct_disponible": numericas.notna().mean().mul(100)
})


print("Resumen de faltantes:")
display(missing_summary.round(3))

Resumen de faltantes:


,n_observado,n_missing,pct_missing,pct_disponible
H1 T (°C),8872,33,0.371,99.629
H2 T (°C),8903,2,0.022,99.978
H3 T (°C),7823,1082,12.150,87.850
H4 T (°C),7603,1302,14.621,85.379
H1 HRCal (%),8872,33,0.371,99.629
H2 HRCal (%),8903,2,0.022,99.978
H3 HRCal (%),7823,1082,12.150,87.850
H4 HRCal (%),7603,1302,14.621,85.379
SMN T (°C),8905,0,0.000,100.000
SMN HR (%),8905,0,0.000,100.000


In [32]:
DISPONIBILIDAD_COLS = [f"{hobo}_disponible" for hobo in HOBOS]


for col, t_col, hr_col in zip(DISPONIBILIDAD_COLS, TEMP_INTERIOR, HR_INTERIOR):

    df_procesado[col] = (
        df_procesado[t_col].notna()
        & df_procesado[hr_col].notna()
    )


df_procesado["todos_hobo_faltantes"] = ~df_procesado[DISPONIBILIDAD_COLS].any(axis=1)

In [33]:
filas = []

for hobo, disp_col, t_col, hr_col in zip(
    HOBOS, DISPONIBILIDAD_COLS, TEMP_INTERIOR, HR_INTERIOR
):

    disponible = df_procesado[disp_col]
    t_falta = df_procesado[t_col].isna()
    hr_falta = df_procesado[hr_col].isna()

    filas.append({
        "sensor": hobo,
        "n_disponible": disponible.sum(),
        "n_faltante": (~disponible).sum(),
        "pct_disponible": disponible.mean() * 100,
        "T_y_HR_missing": (t_falta & hr_falta).sum(),
        "T_presente_HR_missing": (~t_falta & hr_falta).sum(),
        "HR_presente_T_missing": (t_falta & ~hr_falta).sum()
    })


resumen_sensores = pd.DataFrame(filas).set_index("sensor")


# Control: las tres columnas tienen que sumar el total de faltantes de cada sensor. Si no suman, la tabla está mal armada.

diferencia = (
    resumen_sensores["n_faltante"]
    - resumen_sensores[
        ["T_y_HR_missing", "T_presente_HR_missing", "HR_presente_T_missing"]
    ].sum(axis=1)
)

if (diferencia != 0).any():
    raise ValueError(
        "Las tres columnas no suman el total de faltantes en: "
        f"{diferencia[diferencia != 0].index.tolist()}"
    )


print("Auditoría por sensor (consistencia T/HR y disponibilidad):")
display(resumen_sensores.round(3))

Auditoría por sensor (consistencia T/HR y disponibilidad):


,n_disponible,n_faltante,pct_disponible,T_y_HR_missing,T_presente_HR_missing,HR_presente_T_missing
sensor,,,,,,
H1,8872,33,99.629,33,0,0
H2,8903,2,99.978,2,0,0
H3,7823,1082,87.850,1082,0,0
H4,7603,1302,85.379,1302,0,0


In [34]:
COLUMNAS_BLOQUES = ["serie", "inicio", "fin", "n_horas"]


def bloques_faltantes(mask, etiqueta):

    if not mask.any():
        return pd.DataFrame(columns=COLUMNAS_BLOQUES)

    grupo = mask.ne(mask.shift()).cumsum()

    resultado = (
        pd.DataFrame({"fecha": mask.index[mask], "grupo": grupo[mask].values})
        .groupby("grupo")
        .agg(
            inicio=("fecha", "min"),
            fin=("fecha", "max"),
            n_horas=("fecha", "size")
        )
        .reset_index(drop=True)
    )

    resultado.insert(0, "serie", etiqueta)

    return resultado


bloques = (
    [
        bloques_faltantes(~df_procesado[col], hobo)
        for hobo, col in zip(HOBOS, DISPONIBILIDAD_COLS)
    ]
    + [
        bloques_faltantes(df_procesado[col].isna(), col)
        for col in VARIABLES_SMN
    ]
)

bloques = [b for b in bloques if not b.empty]

missing_blocks = (
    pd.concat(bloques, ignore_index=True)
    if bloques
    else pd.DataFrame(columns=COLUMNAS_BLOQUES)
)


print("Bloques consecutivos de faltantes:")
display(missing_blocks)

Bloques consecutivos de faltantes:


,serie,inicio,fin,n_horas
0,H1,2023-03-16 15:00:00,2023-03-16 15:00:00,1
1,H1,2023-06-01 16:00:00,2023-06-01 16:00:00,1
2,H1,2023-08-23 10:00:00,2023-08-24 16:00:00,31
3,H2,2023-03-16 15:00:00,2023-03-16 15:00:00,1
4,H2,2023-06-01 16:00:00,2023-06-01 16:00:00,1
5,H3,2023-03-16 15:00:00,2023-03-16 15:00:00,1
6,H3,2023-06-01 16:00:00,2023-06-01 16:00:00,1
7,H3,2023-08-23 10:00:00,2023-08-24 16:00:00,31
8,H3,2023-09-05 18:00:00,2023-10-19 10:00:00,1049
9,H4,2023-03-16 15:00:00,2023-03-16 15:00:00,1


## 6. Bloques faltantes y controles básicos de consistencia

La extensión de los bloques consecutivos sin observaciones es relevante para decidir qué tramos pueden utilizarse más adelante en análisis que requieren continuidad temporal. También se aplican controles físicos elementales a la humedad relativa. Estos controles generan banderas de auditoría; no reemplazan una calibración instrumental.

In [35]:
HR_COLS = HR_INTERIOR + [HR_SMN]


def nombre_flag_qc(col):

    limpio = col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct")

    return f"qc_{limpio}_fuera_rango"


fuera_de_rango = (df_procesado[HR_COLS] < 0) | (df_procesado[HR_COLS] > 100)

hr_invalid_summary = fuera_de_rango.sum().to_frame("n_fuera_0_100")


print("Valores de HR fuera del rango [0, 100]:")
display(hr_invalid_summary)


if fuera_de_rango.any().any():

    for col in HR_COLS:
        df_procesado[nombre_flag_qc(col)] = fuera_de_rango[col]

    print("Se agregaron banderas qc_* al dataset procesado.")

Valores de HR fuera del rango [0, 100]:


,n_fuera_0_100
H1 HRCal (%),0
H2 HRCal (%),0
H3 HRCal (%),0
H4 HRCal (%),0
SMN HR (%),0


## 7. Limitaciones instrumentales conocidas

Se documentan diferencias de granularidad y cobertura de calibración entre instrumentos. Esta información no se utiliza para modificar automáticamente los valores: queda registrada para interpretar comparaciones posteriores y para definir, cuando sea necesario, pisos de precisión coherentes con la resolución observada.

In [36]:
MODELOS = {
    "H1": "HOBO H08-003-02",
    "H2": "HOBO H08-003-02",
    "H3": "HOBO H08-003-02",
    "H4": "HOBO UX100-003"
}

UBICACIONES = {
    "H1": "muro este, en contacto con el exterior",
    "H2": "estantería hasta abril de 2023 y muro oeste en contacto con el "
          "exterior desde entonces",
    "H3": "muro interno; el techo sobre esa zona no está cubierto por arboleda",
    "H4": "zona central de la reserva, alejada de muros y aberturas"
}

RANGO_CALIBRACION_T = (20, 25)
RANGO_CALIBRACION_HR = (28, 75)


def granularidad_observada(serie):

    unicos = np.sort(serie.dropna().unique())
    distancias = np.diff(unicos)

    return {
        "n_valores": len(unicos),
        "dist_min": distancias.min() if len(distancias) else np.nan,
        "dist_max": distancias.max() if len(distancias) else np.nan
    }


def pct_fuera(serie, rango):

    bajo, alto = rango
    s = serie.dropna()

    return pd.Series({
        "pct_bajo": 100 * (s < bajo).mean(),
        "pct_sobre": 100 * (s > alto).mean(),
        "pct_fuera": 100 * ((s < bajo) | (s > alto)).mean()
    })


granularidad_sensores = pd.DataFrame([
    {
        "sensor": hobo,
        "modelo": MODELOS[hobo],
        **{
            f"T_{k}": v
            for k, v in granularidad_observada(df_procesado[t_col]).items()
        },
        **{
            f"HR_{k}": v
            for k, v in granularidad_observada(df_procesado[hr_col]).items()
        }
    }
    for hobo, t_col, hr_col in zip(HOBOS, TEMP_INTERIOR, HR_INTERIOR)
]).set_index("sensor")


cobertura_calibracion = pd.DataFrame([
    {
        "sensor": hobo,
        **{
            f"T_{k}": v
            for k, v in pct_fuera(
                df_procesado[t_col], RANGO_CALIBRACION_T
            ).items()
        },
        **{
            f"HR_{k}": v
            for k, v in pct_fuera(
                df_procesado[hr_col], RANGO_CALIBRACION_HR
            ).items()
        }
    }
    for hobo, t_col, hr_col in zip(HOBOS, TEMP_INTERIOR, HR_INTERIOR)
]).set_index("sensor")


limitaciones_instrumentales = granularidad_sensores.join(
    cobertura_calibracion
)


print("Granularidad observada (distancias entre valores registrados):")
display(granularidad_sensores.round(3))

print(
    f"Observaciones fuera del rango de calibración "
    f"(T {RANGO_CALIBRACION_T[0]}-{RANGO_CALIBRACION_T[1]} °C, "
    f"HR {RANGO_CALIBRACION_HR[0]}-{RANGO_CALIBRACION_HR[1]} %), en %:"
)
display(cobertura_calibracion.round(1))

Granularidad observada (distancias entre valores registrados):


,modelo,T_n_valores,T_dist_min,T_dist_max,HR_n_valores,HR_dist_min,HR_dist_max
sensor,,,,,,,
H1,HOBO H08-003-02,45,0.39,0.43,357,0.1,1.9
H2,HOBO H08-003-02,42,0.40,0.44,368,0.1,0.8
H3,HOBO H08-003-02,37,0.39,0.43,360,0.1,1.6
H4,HOBO UX100-003,589,0.02,0.05,354,0.1,3.1


Observaciones fuera del rango de calibración (T 20-25 °C, HR 28-75 %), en %:


,T_pct_bajo,T_pct_sobre,T_pct_fuera,HR_pct_bajo,HR_pct_sobre,HR_pct_fuera
sensor,,,,,,
H1,49.7,27.7,77.5,0.0,14.0,14.0
H2,44.2,27.8,71.9,0.0,5.0,5.0
H3,34.1,36.8,70.9,0.0,2.5,2.5
H4,39.9,33.4,73.3,0.0,2.0,2.0


## 8. Eventos y cambios conocidos

Los eventos documentados durante el monitoreo se registran como metadatos de contexto. No se etiquetan a priori como anomalías: primero se detectarán los comportamientos atípicos desde las series y recién después se contrastarán con esta información.

In [37]:
TFI = "Scoccimarro (2024)"

eventos_conocidos = pd.DataFrame([
    {
        "evento": "Interrupción H1 y H3",
        "sensor": "H1; H3",
        "inicio": "2023-08-23 10:00",
        "fin": "2023-08-24 16:00",
        "tipo": "interrupcion",
        "descripcion": "Interrupción documentada por falta de batería.",
        "fuente_evento": TFI,
        "fuente_fecha": "datos_observados",
        "precision_fecha": "horaria",
        "criterio_fecha":
            "Coincide con el bloque faltante observado "
            "en ambos sensores (31 h)."
    },
    {
        "evento": "Interrupción H3",
        "sensor": "H3",
        "inicio": "2023-09-05 18:00",
        "fin": "2023-10-19 10:00",
        "tipo": "interrupcion",
        "descripcion": "Interrupción documentada del registro de H3.",
        "fuente_evento": TFI,
        "fuente_fecha": "datos_observados",
        "precision_fecha": "horaria",
        "criterio_fecha":
            "La fecha de inicio consignada en el TFI presenta "
            "una inconsistencia cronológica, por lo que se "
            "adopta el bloque faltante observado (1049 h)."
    },
    {
        "evento": "Interrupción H4",
        "sensor": "H4",
        "inicio": "2023-07-01 13:00",
        "fin": "2023-08-24 16:00",
        "tipo": "interrupcion",
        "descripcion":
            "Interrupción asociada al apagado accidental "
            "del datalogger.",
        "fuente_evento": TFI,
        "fuente_fecha": "datos_observados",
        "precision_fecha": "horaria",
        "criterio_fecha": "Coincide con el bloque faltante observado (1300 h)."
    },
    {
        "evento": "Reubicación H2",
        "sensor": "H2",
        "inicio": "2023-04-08",
        "fin": "2023-04-14",
        "tipo": "cambio_ubicacion",
        "descripcion":
            "El TFI indica que H2 fue reubicado durante "
            "la segunda semana de abril, pasando de una "
            "estantería a un muro conectado con el exterior. "
            "No se dispone de timestamp exacto ni el evento "
            "deja rastro como bloque faltante.",
        "fuente_evento": TFI,
        "fuente_fecha": "operacionalizacion",
        "precision_fecha": "ventana_semanal",
        "criterio_fecha":
            "Operacionalización de 'segunda semana de abril'. "
            "Estas fechas NO figuran en el TFI: son una ventana "
            "adoptada acá, y cualquier corte que se apoye en "
            "ellas hereda esa incertidumbre."
    },
    {
        "evento": "Muestra 'Reserva visitable'",
        "sensor": "todos",
        "inicio": "2023-05-24",
        "fin": "2023-12-05",
        "tipo": "cambio_ocupacion",
        "descripcion":
            "Se trasladaron 320 obras de la reserva a las salas de planta baja "
            "para su exhibición, con reordenamiento del espacio y mayor "
            "circulación de personal. Cambia la masa higroscópica del ambiente "
            "durante más de seis meses.",
        "fuente_evento": TFI,
        "fuente_fecha": TFI,
        "precision_fecha": "diaria",
        "criterio_fecha": "Fechas consignadas explícitamente en el TFI."
    },
    {
        "evento": "Ventilación programada",
        "sensor": "todos",
        "inicio": "2023-03-01",
        "fin": "2024-01-04",
        "tipo": "rutina_operativa",
        "descripcion":
            "Apertura de puertas y algunas ventanas los lunes y jueves, "
            "aproximadamente de 9 a 18 h. El TFI documenta que la HR interior "
            "sigue a la exterior en esas franjas y permanece plana los fines de "
            "semana y feriados.",
        "fuente_evento": TFI,
        "fuente_fecha": TFI,
        "precision_fecha": "mensual_aproximada",
        "criterio_fecha":
            "El TFI indica que la rutina comenzó 'a partir de marzo' sin "
            "precisar el día; el fin se toma como el final del monitoreo."
    }
])


print("Eventos conocidos:")
display(eventos_conocidos)

Eventos conocidos:


,evento,sensor,inicio,fin,tipo,descripcion,fuente_evento,fuente_fecha,precision_fecha,criterio_fecha
0,Interrupción H1 y H3,H1; H3,2023-08-23 10:00,2023-08-24 16:00,interrupcion,Interrupción documentada por falta de batería.,Scoccimarro (2024),datos_observados,horaria,Coincide con el bloque faltante observado en a...
1,Interrupción H3,H3,2023-09-05 18:00,2023-10-19 10:00,interrupcion,Interrupción documentada del registro de H3.,Scoccimarro (2024),datos_observados,horaria,La fecha de inicio consignada en el TFI presen...
2,Interrupción H4,H4,2023-07-01 13:00,2023-08-24 16:00,interrupcion,Interrupción asociada al apagado accidental de...,Scoccimarro (2024),datos_observados,horaria,Coincide con el bloque faltante observado (130...
3,Reubicación H2,H2,2023-04-08,2023-04-14,cambio_ubicacion,El TFI indica que H2 fue reubicado durante la ...,Scoccimarro (2024),operacionalizacion,ventana_semanal,Operacionalización de 'segunda semana de abril...
4,Muestra 'Reserva visitable',todos,2023-05-24,2023-12-05,cambio_ocupacion,Se trasladaron 320 obras de la reserva a las s...,Scoccimarro (2024),Scoccimarro (2024),diaria,Fechas consignadas explícitamente en el TFI.
5,Ventilación programada,todos,2023-03-01,2024-01-04,rutina_operativa,Apertura de puertas y algunas ventanas los lun...,Scoccimarro (2024),Scoccimarro (2024),mensual_aproximada,El TFI indica que la rutina comenzó 'a partir ...


## 9. Diccionario y resumen de la auditoría

Se consolida un diccionario de variables y un resumen de los controles realizados. Esta sección funciona como punto de cierre del preprocesamiento y como referencia para los notebooks analíticos siguientes.

In [38]:
def doc(unidad, origen, descripcion):
    return {"unidad": unidad, "origen": origen, "descripcion": descripcion}


DESCRIPCIONES_COLUMNAS = {}


for hobo, t_col, hr_col, disp_col in zip(
    HOBOS, TEMP_INTERIOR, HR_INTERIOR, DISPONIBILIDAD_COLS
):

    DESCRIPCIONES_COLUMNAS[t_col] = doc(
        "°C", "medicion",
        f"Temperatura registrada por el datalogger {hobo} ({MODELOS[hobo]}), "
        f"ubicado en {UBICACIONES[hobo]}."
    )

    DESCRIPCIONES_COLUMNAS[hr_col] = doc(
        "%", "medicion_calibrada",
        f"Humedad relativa registrada por el datalogger {hobo} "
        f"({MODELOS[hobo]}), con la corrección de calibración aplicada "
        "previamente al dataset recibido."
    )

    DESCRIPCIONES_COLUMNAS[disp_col] = doc(
        "-", "derivada",
        f"True cuando {hobo} tiene simultáneamente temperatura y humedad "
        "relativa registradas en esa hora."
    )


# Banderas de control de rango: se registran con el nombre exacto que genera
# nombre_flag_qc(). Solo existen en el dataset si hubo valores fuera de rango,
# pero documentarlas siempre mantiene el diccionario sincronizado con el
# código que las crea.

for col in HR_COLS:

    DESCRIPCIONES_COLUMNAS[nombre_flag_qc(col)] = doc(
        "-", "derivada",
        f"Bandera de control: True cuando '{col}' queda fuera del rango "
        "físico [0, 100]."
    )


DESCRIPCIONES_COLUMNAS.update({

    TEMP_SMN: doc(
        "°C", "medicion",
        "Temperatura exterior registrada por la estación del Servicio "
        "Meteorológico Nacional."
    ),

    HR_SMN: doc(
        "%", "medicion",
        "Humedad relativa exterior registrada por la estación del Servicio "
        "Meteorológico Nacional."
    ),

    PREC_SMN: doc(
        "mm", "medicion",
        "Precipitación acumulada en 6 horas. Solo se registra en las horas "
        "03, 09, 15 y 21; las filas intermedias contienen 0 y no son "
        "observaciones independientes."
    ),

    "todos_hobo_faltantes": doc(
        "-", "derivada",
        "True cuando ningún HOBO tiene medición en esa hora."
    ),

    "registro_original": doc(
        "-", "derivada",
        "False cuando la fila no existía en el archivo y fue creada al "
        "completar la grilla horaria. Solo se exporta si alguna fila fue "
        "agregada."
    )
})


filas_diccionario = [{
    "columna": df_procesado.index.name,
    "dtype": str(df_procesado.index.dtype),
    **doc("-", "indice", "Marca temporal horaria del registro.")
}]


for col in df_procesado.columns:

    if col not in DESCRIPCIONES_COLUMNAS:
        raise KeyError(f"Columna sin documentar en el diccionario: {col}")

    filas_diccionario.append({
        "columna": col,
        "dtype": str(df_procesado[col].dtype),
        **DESCRIPCIONES_COLUMNAS[col]
    })


diccionario_columnas = pd.DataFrame(filas_diccionario)


print("Diccionario de columnas:")
display(diccionario_columnas)

Diccionario de columnas:


,columna,dtype,unidad,origen,descripcion
0,fecha,datetime64[us],-,indice,Marca temporal horaria del registro.
1,H1 T (°C),float64,°C,medicion,Temperatura registrada por el datalogger H1 (H...
2,H1 HRCal (%),float64,%,medicion_calibrada,Humedad relativa registrada por el datalogger ...
3,H2 T (°C),float64,°C,medicion,Temperatura registrada por el datalogger H2 (H...
4,H2 HRCal (%),float64,%,medicion_calibrada,Humedad relativa registrada por el datalogger ...
5,H3 T (°C),float64,°C,medicion,Temperatura registrada por el datalogger H3 (H...
6,H3 HRCal (%),float64,%,medicion_calibrada,Humedad relativa registrada por el datalogger ...
7,H4 T (°C),float64,°C,medicion,Temperatura registrada por el datalogger H4 (H...
8,H4 HRCal (%),float64,%,medicion_calibrada,Humedad relativa registrada por el datalogger ...
9,SMN T (°C),float64,°C,medicion,Temperatura exterior registrada por la estació...


In [39]:
metricas = {
    "n_filas_leidas": len(df_copy),
    "n_filas_grilla_horaria": len(df_procesado),
    "fecha_inicio": df_procesado.index.min(),
    "fecha_fin": df_procesado.index.max(),
    "timestamps_duplicados": n_duplicados,
    "timestamps_faltantes": len(fechas_faltantes),
    "horas_sin_ningun_hobo": int(df_procesado["todos_hobo_faltantes"].sum())
}

resumen_preprocessing = pd.DataFrame(metricas.items(), columns=["metrica", "valor"])


print("Resumen del preprocesamiento:")
display(resumen_preprocessing)

Resumen del preprocesamiento:


,metrica,valor
0,n_filas_leidas,8905
1,n_filas_grilla_horaria,8905
2,fecha_inicio,2022-12-29 14:00:00
3,fecha_fin,2024-01-04 14:00:00
4,timestamps_duplicados,0
5,timestamps_faltantes,0
6,horas_sin_ningun_hobo,2


## 10. Exportación

Se guarda el conjunto de datos procesado y los metadatos necesarios para reproducir las decisiones de preparación. Antes de utilizar una versión definitiva conviene ejecutar el notebook completo desde un kernel limpio y comprobar que todas las celdas finalicen sin errores.

In [40]:
METADATOS = {
    "diccionario_columnas.csv": diccionario_columnas,
    "eventos_conocidos.csv": eventos_conocidos,
    "missing_blocks.csv": missing_blocks,
    "limitaciones_instrumentales.csv": limitaciones_instrumentales.reset_index()
}


for nombre, tabla in METADATOS.items():

    tabla.to_csv(METADATA_DIR / nombre, index=False)

    print(f"{str(METADATA_DIR / nombre):<50} {len(tabla):>4} filas")

..\data\metadata\diccionario_columnas.csv            17 filas
..\data\metadata\eventos_conocidos.csv                6 filas
..\data\metadata\missing_blocks.csv                  12 filas
..\data\metadata\limitaciones_instrumentales.csv      4 filas


In [41]:
CSV_PROCESSED_PATH = PROCESSED_DIR / "sivori_horario_procesado.csv"

df_procesado.to_csv(CSV_PROCESSED_PATH, encoding="utf-8")


filas, columnas = df_procesado.shape

print(f"{str(CSV_PROCESSED_PATH):<50} {filas:>4} filas x {columnas} columnas")

..\data\processed\sivori_horario_procesado.csv     8905 filas x 16 columnas


## Referencia del conjunto de datos y del contexto de monitoreo

Scoccimarro, M. (2024). Implementación de un sistema de monitoreo ambiental orientado al diseño de estrategias sostenibles para la conservación preventiva de colecciones patrimoniales: Caso de estudio: Reserva Infanta Isabel del Museo de Artes Plásticas Eduardo Sívori [Tesina de licenciatura]. Universidad Nacional de San Martín. https://ri.unsam.edu.ar/handle/123456789/3274

Esta referencia documenta el sistema de monitoreo y constituye el antecedente directo del conjunto de datos. Las referencias estadísticas específicas se incorporarán en los notebooks donde se apliquen las técnicas correspondientes.